In [1]:
!pip install pytorch-tabnet lightning -q

In [1]:
from google.colab import files
uploaded = files.upload()   # upload src\utils.py and data\raw_close_prices.csv

Saving raw_close_prices.csv to raw_close_prices.csv


In [4]:
import tempfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
import keras
from keras import layers, regularizers
from torch.utils.data import DataLoader, TensorDataset
from pytorch_tabnet.tab_model import TabNetRegressor
from fastai.tabular.all import *
from sklearn.preprocessing import StandardScaler
from utils import evaluate

# Lightning imports go AFTER the fastai star import, so fastai's own names don't overwrite them
import lightning as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

warnings.filterwarnings("ignore")

close = pd.read_csv("raw_close_prices.csv", index_col=0, parse_dates=True)
close = close.dropna(subset=["MSFT"]).ffill(limit=3).dropna()
weekly_close = close.resample("W-FRI").last()
if close.index[-1] < weekly_close.index[-1]:
    weekly_close = weekly_close.iloc[:-1]
weekly_ret = weekly_close.pct_change().dropna()

lags = [0, 1, 2, 3, 4]
features = pd.concat([weekly_ret.shift(l).add_suffix(f"_lag{l}") for l in lags], axis=1, sort=False)
x_next = features.iloc[[-1]]
print("Latest complete week:", weekly_ret.index[-1].date())

def make_xy(ticker):
    target = weekly_ret[ticker].shift(-1)
    d = pd.concat([features, target.rename("y")], axis=1, sort=False).dropna()
    return d.drop(columns="y"), d["y"]

Latest complete week: 2026-10-23


In [5]:
def train_keras(Xs, ys, Xv, yv, Xp, seed):
    keras.utils.set_random_seed(seed)
    m = keras.Sequential([
        layers.Input(shape=(Xs.shape[1],)),
        layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(1e-3)),
        layers.Dropout(0.3),
        layers.Dense(16, activation="relu", kernel_regularizer=regularizers.l2(1e-3)),
        layers.Dropout(0.3),
        layers.Dense(1),
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    m.fit(Xs, ys, validation_data=(Xv, yv), epochs=300, batch_size=32, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=20,
                                                   restore_best_weights=True)])
    return m.predict(Xp, verbose=0).ravel()


def train_fastai(Xs, ys, Xv, yv, Xp, seed):
    cols = [f"f{i}" for i in range(Xs.shape[1])]
    df = pd.DataFrame(np.vstack([Xs, Xv]), columns=cols)
    df["target"] = np.concatenate([ys, yv])
    n = len(Xs)
    to = TabularPandas(df, procs=[], cont_names=cols, y_names="target",
                       y_block=RegressionBlock(),
                       splits=(list(range(n)), list(range(n, len(df)))))
    dls = to.dataloaders(bs=32)
    set_seed(seed, reproducible=True)
    learn = tabular_learner(dls, layers=[32, 16], config=tabular_config(ps=[0.3, 0.3]),
                            wd=0.1, loss_func=MSELossFlat())
    with learn.no_bar(), learn.no_logging():
        learn.fit_one_cycle(40, 1e-3, cbs=[
            EarlyStoppingCallback(monitor="valid_loss", patience=8),
            SaveModelCallback(monitor="valid_loss", fname=f"tmp_{seed}")])
    dl = dls.test_dl(pd.DataFrame(Xp, columns=cols).assign(target=0.0))
    p, _ = learn.get_preds(dl=dl)
    return p.numpy().ravel()


def train_tabnet(Xs, ys, Xv, yv, Xp, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    m = TabNetRegressor(n_d=8, n_a=8, n_steps=3, gamma=1.5, lambda_sparse=1e-3,
                        optimizer_params=dict(lr=2e-3, weight_decay=1e-3),
                        seed=seed, verbose=0)
    m.fit(Xs, ys.reshape(-1, 1), eval_set=[(Xv, yv.reshape(-1, 1))], eval_metric=["rmse"],
          max_epochs=200, patience=20, batch_size=64, virtual_batch_size=32)
    return m.predict(Xp).ravel()


class MLP(pl.LightningModule):
    def __init__(self, n_features):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(
            nn.Linear(n_features, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.3), nn.Linear(16, 1))
        self.loss = nn.MSELoss()
    def forward(self, x):
        return self.net(x).squeeze(-1)
    def training_step(self, b, _):
        x, y = b
        return self.loss(self(x), y)
    def validation_step(self, b, _):
        x, y = b
        self.log("val_loss", self.loss(self(x), y))
    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-2)


def _dl(X, y, shuffle):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=32, shuffle=shuffle)


def train_lightning(Xs, ys, Xv, yv, Xp, seed):
    pl.seed_everything(seed, workers=True)
    model = MLP(Xs.shape[1])
    ckpt = ModelCheckpoint(dirpath=tempfile.mkdtemp(), monitor="val_loss", mode="min", save_top_k=1)
    tr = pl.Trainer(max_epochs=200, accelerator="auto", logger=False,
                    enable_progress_bar=False, enable_model_summary=False,
                    callbacks=[EarlyStopping(monitor="val_loss", patience=20), ckpt])
    tr.fit(model, _dl(Xs, ys, True), _dl(Xv, yv, False))
    best = MLP.load_from_checkpoint(ckpt.best_model_path)
    best.eval()
    with torch.no_grad():
        return best(torch.tensor(Xp, dtype=torch.float32)).numpy()


FRAMEWORKS = {"Keras MLP": train_keras, "fastai tabular": train_fastai,
              "TabNet": train_tabnet, "Lightning MLP": train_lightning}

In [6]:
SEEDS = range(3)

def fit_predict(fn, X, y, Xp):
    """Fit on X,y (last 15% = chronological validation); return predictions for Xp in return units."""
    n_val = int(len(X) * 0.15)
    X_sub, X_val = X.iloc[:-n_val], X.iloc[-n_val:]
    y_sub, y_val = y.iloc[:-n_val], y.iloc[-n_val:]
    sc = StandardScaler().fit(X_sub)
    mu, sd = y_sub.mean(), y_sub.std()
    ys, yv = ((y_sub - mu) / sd).values, ((y_val - mu) / sd).values
    preds = [fn(sc.transform(X_sub), ys, sc.transform(X_val), yv, sc.transform(Xp), s)
             for s in SEEDS]
    return np.mean(preds, axis=0) * sd + mu


rows, forecasts = [], []
for ticker in ["MSFT", "IBM", "GOOGL"]:
    X, y = make_xy(ticker)
    split = int(len(X) * 0.8)
    X_tr, X_te, y_tr, y_te = X.iloc[:split], X.iloc[split:], y.iloc[:split], y.iloc[split:]
    mu_tr = y_tr.mean()

    for name, fn in FRAMEWORKS.items():
        p_test = fit_predict(fn, X_tr, y_tr, X_te)
        rows.append({**evaluate(y_te, p_test, mu_tr, name), "stock": ticker,
                     "up_share_test": (y_te > 0).mean()})
        p_next = float(fit_predict(fn, X, y, x_next)[0])
        forecasts.append({"stock": ticker, "model": name,
                          "expected_weekly_return": p_next,
                          "annualized_approx": p_next * 52,
                          "historical_mean_weekly": y.mean()})
        print(ticker, name, "done", flush=True)

metrics = pd.DataFrame(rows).set_index(["stock", "model"])
fc = pd.DataFrame(forecasts).set_index(["stock", "model"])
print(metrics.round(4))
print((fc * 100).round(3))
metrics.to_csv("deep_multi_stock_metrics.csv")
fc.to_csv("deep_expected_returns.csv")

MSFT Keras MLP done
Better model found at epoch 0 with valid_loss value: 1.793670892715454.
Better model found at epoch 5 with valid_loss value: 1.7755656242370605.
Better model found at epoch 6 with valid_loss value: 1.7327463626861572.
Better model found at epoch 7 with valid_loss value: 1.7162249088287354.
Better model found at epoch 8 with valid_loss value: 1.6983433961868286.
Better model found at epoch 9 with valid_loss value: 1.6689785718917847.
Better model found at epoch 11 with valid_loss value: 1.6292879581451416.
No improvement since epoch 11: early stopping


Better model found at epoch 0 with valid_loss value: 1.6987653970718384.
No improvement since epoch 0: early stopping


Better model found at epoch 0 with valid_loss value: 1.7090871334075928.
Better model found at epoch 8 with valid_loss value: 1.6780269145965576.
Better model found at epoch 9 with valid_loss value: 1.6312737464904785.
Better model found at epoch 14 with valid_loss value: 1.6233711242675781.
Better model found at epoch 18 with valid_loss value: 1.6227223873138428.
Better model found at epoch 20 with valid_loss value: 1.6162376403808594.
No improvement since epoch 20: early stopping


Better model found at epoch 0 with valid_loss value: 1.719734787940979.
Better model found at epoch 4 with valid_loss value: 1.7118418216705322.
Better model found at epoch 5 with valid_loss value: 1.6832953691482544.
Better model found at epoch 6 with valid_loss value: 1.6737056970596313.
Better model found at epoch 7 with valid_loss value: 1.6441317796707153.
Better model found at epoch 8 with valid_loss value: 1.598542332649231.
Better model found at epoch 9 with valid_loss value: 1.5957672595977783.
Better model found at epoch 10 with valid_loss value: 1.5826648473739624.
Better model found at epoch 13 with valid_loss value: 1.578747272491455.
Better model found at epoch 14 with valid_loss value: 1.5728321075439453.
Better model found at epoch 21 with valid_loss value: 1.5673775672912598.
No improvement since epoch 21: early stopping


Better model found at epoch 0 with valid_loss value: 1.656786322593689.
Better model found at epoch 5 with valid_loss value: 1.6299591064453125.
Better model found at epoch 6 with valid_loss value: 1.5775996446609497.
Better model found at epoch 8 with valid_loss value: 1.5720829963684082.
Better model found at epoch 9 with valid_loss value: 1.5581207275390625.
Better model found at epoch 15 with valid_loss value: 1.5471769571304321.
Better model found at epoch 22 with valid_loss value: 1.5448925495147705.
Better model found at epoch 24 with valid_loss value: 1.539942979812622.
Better model found at epoch 27 with valid_loss value: 1.5339043140411377.
Better model found at epoch 28 with valid_loss value: 1.5285041332244873.
Better model found at epoch 29 with valid_loss value: 1.5257773399353027.
No improvement since epoch 29: early stopping


...

Better model found at epoch 0 with valid_loss value: 1.365157961845398.
Better model found at epoch 4 with valid_loss value: 1.349482536315918.
Better model found at epoch 5 with valid_loss value: 1.328798770904541.
Better model found at epoch 6 with valid_loss value: 1.2861870527267456.
Better model found at epoch 7 with valid_loss value: 1.2775027751922607.
Better model found at epoch 10 with valid_loss value: 1.2724231481552124.
Better model found at epoch 12 with valid_loss value: 1.2681350708007812.
Better model found at epoch 13 with valid_loss value: 1.2654799222946167.
No improvement since epoch 13: early stopping


Better model found at epoch 0 with valid_loss value: 1.4403610229492188.
Better model found at epoch 5 with valid_loss value: 1.4047991037368774.
Better model found at epoch 6 with valid_loss value: 1.3885786533355713.
Better model found at epoch 7 with valid_loss value: 1.3789399862289429.
Better model found at epoch 8 with valid_loss value: 1.370713472366333.
Better model found at epoch 9 with valid_loss value: 1.343986988067627.
Better model found at epoch 10 with valid_loss value: 1.3198497295379639.
Better model found at epoch 11 with valid_loss value: 1.305681586265564.
Better model found at epoch 13 with valid_loss value: 1.3028831481933594.
Better model found at epoch 15 with valid_loss value: 1.291144847869873.
Better model found at epoch 16 with valid_loss value: 1.2873698472976685.
Better model found at epoch 17 with valid_loss value: 1.2783743143081665.
Better model found at epoch 18 with valid_loss value: 1.2758564949035645.
Better model found at epoch 25 with valid_loss v

GOOGL fastai tabular done

Early stopping occurred at epoch 36 with best_epoch = 16 and best_val_0_rmse = 1.28351

Early stopping occurred at epoch 24 with best_epoch = 4 and best_val_0_rmse = 1.24104

Early stopping occurred at epoch 22 with best_epoch = 2 and best_val_0_rmse = 1.29163

Early stopping occurred at epoch 32 with best_epoch = 12 and best_val_0_rmse = 1.13702

Early stopping occurred at epoch 23 with best_epoch = 3 and best_val_0_rmse = 1.18377

Early stopping occurred at epoch 52 with best_epoch = 32 and best_val_0_rmse = 1.14554
GOOGL TabNet done


INFO: Seed set to 0
INFO:lightning.fabric.utilities.seed:Seed set to 0
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: Seed set to 1
INFO:lightning.fabric.utilities.seed:Seed set to 1
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: Fal

GOOGL Lightning MLP done
                        RMSE     MAE      R2  R2_OOS  DirAcc  up_share_test
stock model                                                                
MSFT  Keras MLP       0.0364  0.0262  0.0154  0.0166  0.4971         0.5200
      fastai tabular  0.0363  0.0261  0.0218  0.0230  0.5086         0.5200
      TabNet          0.0376  0.0270 -0.0483 -0.0470  0.5029         0.5200
      Lightning MLP   0.0365  0.0261  0.0110  0.0122  0.5029         0.5200
IBM   Keras MLP       0.0470  0.0318 -0.0169 -0.0100  0.4971         0.5771
      fastai tabular  0.0468  0.0321 -0.0100 -0.0031  0.5257         0.5771
      TabNet          0.0468  0.0319 -0.0069 -0.0001  0.4914         0.5771
      Lightning MLP   0.0468  0.0316 -0.0104 -0.0036  0.6000         0.5771
GOOGL Keras MLP       0.0429  0.0332 -0.0039 -0.0004  0.5829         0.5600
      fastai tabular  0.0432  0.0338 -0.0221 -0.0185  0.5086         0.5600
      TabNet          0.0439  0.0338 -0.0537 -0.0500  0.5086   

In [7]:
from google.colab import files
for f in ["deep_multi_stock_metrics.csv", "deep_expected_returns.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>